# UHI Modelling V2

I am attempting a revised workflow for modelling UHI.
First, the model is now going to include weather variables obtained from OpenMeteo. The data will then be aggregated with the Landsat data and used to build the model with the workflow below:  
1. Identify the world's climates from an authority (like the Trewartha climate classification).
2. Get substantial satellite (which will eventually include those those climate categories) and weather data on countries in those regions across the months (seasons) in a particular year.
3. Train a model on the data, implementing train-validation-test split, and predict UHI intensity on the test set.
4. Group the test examples by climate and compute the RMSE scores across the climates.
5. Cluster the UHI features to identify the types and categories of UHI.
6. Then I, human, assess the errors and prediction accuracies across each manufactured UHI cluster and how they vary across climates.

***"This project predicts urban heat island intensity using satellite and environmental data and evaluates how prediction reliability and errors vary across global climates and urban thermal types."***

## Testing aggregation with minimal data

To test the flow of data:  
1. From the various countries
2. In the various climates
3. Once a week from Jan 1 2025 to Dec 31 2025
4. From Earth Engine (Landsat) and OpenMeteo

I will be using as minimal data as possible. This will look like:  
1. Lagos, Ontario, Helsinki, and Tehran
2. Aw, Dc, Dcb, Bsk
3. Once a month Jan 1 2025 to Dec 31 2025
4. From Earth Engine and OpenMeteo

## Fetching from Open-Meteo

Again, the cities I want to sample are:
1. Lagos, Nigeria
2. Ontario, Canada
3. Tehran, Iran
4. Helsinki, Finland

The variables I am getting from Open-meteo are:
1. Cloud Cover (Low)
2. Air Temperature (2m)
3. Ralative Humidity
4. Precipitation
5. Wind Speed (10m)

In [1]:
import ee
import geopandas as gpd

ee.Authenticate()
ee.Initialize()

In [2]:
from spectral import get_spectral
from weather import process_city_weather

cities = [
    {"code": "CAN", "name": "Ontario"},
    {"code": "IRN", "name": "Tehran"},
    {"code": "NGA", "name": "Lagos"},
	{"code": "FIN", "name": "Uusimaa"},
]

date_range = ("2025-01-01", "2025-12-31")

for city in cities:
    get_spectral(city['code'], city['name'], date_range)
    # process_city_weather(city['code'], date_range)


c:\Software Projects\Data Projects\uhi-modelling\uhenv\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Data has already been processed at data/CAN_spectral_features.csv.

Data has already been processed at data/IRN_spectral_features.csv.

Data has already been processed at data/NGA_spectral_features.csv.

Data has already been processed at data/FIN_spectral_features.csv.



In [3]:
import pandas as pd

for city in cities:
    df = pd.read_csv(f"data/{city['code']}_spectral_features.csv")
    urban = (df["LandCover"] == 50).sum()
    rural = (df["LandCover"] != 50).sum()
    print(f"{city['name']}: {urban} urban, {rural} rural, total: {len(df)}")

Ontario: 100 urban, 100 rural, total: 200
Tehran: 100 urban, 100 rural, total: 200
Lagos: 100 urban, 100 rural, total: 200
Uusimaa: 100 urban, 100 rural, total: 200


In [4]:
for city in cities:
    process_city_weather(city['code'], date_range)

Data has already been processed at data/CAN_Full_UHI_Data.csv.

Data has already been processed at data/IRN_Full_UHI_Data.csv.

Data has already been processed at data/NGA_Full_UHI_Data.csv.

Data has already been processed at data/FIN_Full_UHI_Data.csv.



Now that we have fetched our data, we will add the `city` column and concatenate all four datasets.

In [5]:
import pandas as pd

dfs = []
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_Full_UHI_Data.csv")
    df["city"] = city["name"]
    dfs.append(df)

uhi_data = pd.concat(dfs, ignore_index=True)

print(uhi_data.shape)
print(uhi_data["city"].value_counts())

(28800, 17)
city
Ontario    7200
Tehran     7200
Lagos      7200
Uusimaa    7200
Name: count, dtype: int64


In [6]:
print(uhi_data.head())
print(uhi_data.info())

                        date  humidity  precipitation  wind_speed  \
0  2025-01-01 08:00:00+00:00  94.77039            0.0   16.428390   
1  2025-01-01 14:00:00+00:00  92.03161            0.7   24.316660   
2  2025-01-01 20:00:00+00:00  78.35640            0.1   26.810333   
3  2025-02-01 08:00:00+00:00  72.55278            0.0   22.528780   
4  2025-02-01 14:00:00+00:00  70.43306            0.0   15.568700   

   cloud_cover_low  air_temperature      Albedo   Elevation        LST  \
0            100.0             1.90  12868.1422  179.384567  285.04745   
1             96.0             1.10  12868.1422  179.384567  285.04745   
2             79.0             1.20  12868.1422  179.384567  285.04745   
3              0.0           -12.90  12868.1422  179.384567  285.04745   
4              0.0           -15.75  12868.1422  179.384567  285.04745   

   LandCover     MNDWI      NDBI      NDVI      SAVI   latitude  longitude  \
0       50.0 -0.113918 -0.061446  0.165858  0.248782  43.96604

Let's clean up the data:
1. Convert date to datetime
2. Convert city to category
3. Convert LandCover to category

In [7]:
uhi_data["date"] = pd.to_datetime(uhi_data["date"], utc = True)

uhi_data.loc[uhi_data["date"].dt.hour.isin(range(6, 11)), "time"] = "morning"
uhi_data.loc[uhi_data["date"].dt.hour.isin(range(12, 15)), "time"] = "afternoon"
uhi_data.loc[uhi_data["date"].dt.hour.isin(range(16, 23)), "time"] = "evening"
    
print(uhi_data["time"].value_counts())
print(uhi_data["time"].isnull().sum())

time
morning      9600
afternoon    9600
evening      9600
Name: count, dtype: int64
0


I am going to experiment with clustering the points to identify the inherent climate classifications they belong to, because I cannot find a fine-grained Trewartha classification.

In [ ]:
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans

# pixel_profiles = uhi_data.groupby(["latitude", "longitude"]).agg(
#     mean_temp=("air_temperature", "mean"),
#     mean_humidity=("humidity", "mean"),
#     total_precip=("precipitation", "sum"),
#     mean_wind=("wind_speed", "mean"),
#     mean_cloud=("cloud_cover_low", "mean")
# ).reset_index()

# features = ["mean_temp", "mean_humidity", "total_precip", "mean_wind", "mean_cloud"]

# scaler = StandardScaler()
# scaled = scaler.fit_transform(pixel_profiles[features])


# inertias = []
# k_range = range(2, 11)
# for k in k_range:
#     km = KMeans(n_clusters=k, random_state=42)
#     km.fit(scaled)
#     inertias.append(km.inertia_)

# import matplotlib.pyplot as plt
# plt.plot(k_range, inertias, marker="o")
# plt.xlabel("Number of clusters")
# plt.ylabel("Inertia")
# plt.title("Elbow Method")
# plt.show()

# print(intertias)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pixel_profiles = uhi_data.groupby(["latitude", "longitude"]).agg(
    mean_temp=("air_temperature", "mean"),
    mean_humidity=("humidity", "mean"),
    total_precip=("precipitation", "sum"),
    mean_wind=("wind_speed", "mean"),
    mean_cloud=("cloud_cover_low", "mean")
).reset_index()

features = ["mean_temp", "mean_humidity", "total_precip", "mean_wind", "mean_cloud"]
scaler = StandardScaler()
scaled = scaler.fit_transform(pixel_profiles[features])

km = KMeans(n_clusters=4, random_state=42)
pixel_profiles["climate"] = km.fit_predict(scaled)

uhi_data = uhi_data.merge(
    pixel_profiles[["latitude", "longitude", "climate"]],
    on=["latitude", "longitude"],
    how="left"
)

print(uhi_data["climate"].value_counts())
print(uhi_data["climate"].isnull().sum())

climate
1    11556
0     7200
2     7200
3     2844
Name: count, dtype: int64
0


Now, deriving SUHI and AUHI.

In [12]:
uhi_data["is_urban"] = (uhi_data["LandCover"] == 50).astype(int)

# Separate urban and rural
urban = uhi_data[uhi_data["is_urban"] == 1].groupby(["city", "date", "time"]).agg(
    LST_urban=("LST", "mean"),
    airtemp_urban=("air_temperature", "mean"),
    n_urban=("LST", "count")
).reset_index()

rural = uhi_data[uhi_data["is_urban"] == 0].groupby(["city", "date", "time"]).agg(
    LST_rural=("LST", "mean"),
    airtemp_rural=("air_temperature", "mean"),
    n_rural=("LST", "count")
).reset_index()

# Merge and compute UHI intensity
uhi_intensity = urban.merge(rural, on=["city", "date", "time"], how="inner")
uhi_intensity["SUHI"] = uhi_intensity["LST_urban"] - uhi_intensity["LST_rural"]
uhi_intensity["AUHI"] = uhi_intensity["airtemp_urban"] - uhi_intensity["airtemp_rural"]

print(uhi_intensity.shape)
print(uhi_intensity[["SUHI", "AUHI"]].describe())
print(uhi_intensity[["n_urban", "n_rural"]].value_counts())

(144, 11)
             SUHI        AUHI
count  144.000000  144.000000
mean     1.055471    1.150687
std      1.344073    2.010427
min     -0.562931   -6.276605
25%     -0.015930   -0.072750
50%      0.958839    0.358375
75%      2.030240    2.989625
max      2.867138    7.194395
n_urban  n_rural
100      100        144
Name: count, dtype: int64


In [18]:
print(uhi_intensity[uhi_intensity["city"] == "Ontario"].head())

       city                      date       time   LST_urban  airtemp_urban  \
36  Ontario 2025-01-01 08:00:00+00:00    morning  284.912131       1.648435   
37  Ontario 2025-01-01 14:00:00+00:00  afternoon  284.912131       1.512435   
38  Ontario 2025-01-01 20:00:00+00:00    evening  284.912131       2.072435   
39  Ontario 2025-02-01 08:00:00+00:00    morning  284.912131     -11.629565   
40  Ontario 2025-02-01 14:00:00+00:00  afternoon  284.912131     -14.187565   

    n_urban   LST_rural  airtemp_rural  n_rural      SUHI      AUHI  
36      100  282.044993       -0.55296      100  2.867138  2.201395  
37      100  282.044993        0.85654      100  2.867138  0.655895  
38      100  282.044993        1.09154      100  2.867138  0.980895  
39      100  282.044993      -12.58896      100  2.867138  0.959395  
40      100  282.044993      -14.59346      100  2.867138  0.405895  


In [18]:
uhi_data[uhi_data["is_urban"] == 1].groupby("city").size()

city
Lagos      1620
Ontario      36
Tehran      396
Uusimaa     432
dtype: int64